# Migration — rename deprecated task statuses (2026 status rework)Bulk-updates the `task` layer in `datateam_portfolio_v2` so every record usesthe canonical 2026 task-status vocabulary:| Old status     | →  | New status ||----------------|----|------------|| `In Progress`  | →  | `Active`   || `Not Started`  | →  | `Planned`  || `Pending`      | →  | `Planned`  || `Completed`    | →  | `Complete` |**Run this notebook once at deploy time** of v1.72.0.0. The app's`agolTaskToLocal` already translates these values on load, so the app keepsworking with mixed data — but this notebook permanently fixes the source sothe deprecated values stop appearing in raw queries, exports, and reports.You must be **owner or admin** of the `datateam_portfolio_v2` service.Idempotent — re-running after the migration just reports zero rows touched.

In [ ]:
from arcgis.gis import GIS
from arcgis.features import FeatureLayer

gis = GIS("home")
print(f"Signed in as {gis.users.me.username} @ {gis.url}")

TASK_LAYER_URL = "https://services3.arcgis.com/9coHY2fvuFjG9HQX/ArcGIS/rest/services/datateam_portfolio_v2/FeatureServer/1"
tasks = FeatureLayer(TASK_LAYER_URL, gis)
print("Task layer:", tasks.properties.get('name', '(unknown)'))

## Step 1 — Audit current statusesCounts every distinct status value present in the task layer. Shows you what'sout there before any writes happen. If the only deprecated values that appearare `In Progress`, `Not Started`, `Pending`, and/or `Completed`, you're good.If something unexpected appears (a typo, a status from a different convention),review it before proceeding.

In [ ]:
from collections import Counter

all_rows = tasks.query(where="deleted_at IS NULL", out_fields="OBJECTID,status,title", return_geometry=False).features
print(f"Total active (non-deleted) tasks: {len(all_rows)}")
print()

counts = Counter(f.attributes.get('status') or '(blank)' for f in all_rows)
print("Current status distribution:")
for status, n in sorted(counts.items(), key=lambda kv: (-kv[1], kv[0])):
    print(f"  {n:5d}  {status}")

## Step 2 — Preview the migrationThe mapping below is the canonical migration table. The next cell shows howmany tasks would be touched, and lists up to 10 examples per category so youcan spot-check before the write.

In [ ]:
STATUS_MIGRATIONS = {
    'In Progress': 'Active',
    'Not Started': 'Planned',
    'Pending':     'Planned',
    'Completed':   'Complete',
}

to_update = []
samples_by_old = {old: [] for old in STATUS_MIGRATIONS}
counts_by_old = Counter()

for f in all_rows:
    old = f.attributes.get('status')
    if old not in STATUS_MIGRATIONS:
        continue
    new = STATUS_MIGRATIONS[old]
    to_update.append({'attributes': {'OBJECTID': f.attributes['OBJECTID'], 'status': new}})
    counts_by_old[old] += 1
    if len(samples_by_old[old]) < 10:
        samples_by_old[old].append((f.attributes.get('title') or '(no title)', f.attributes['OBJECTID']))

print(f"Tasks that will be migrated: {len(to_update)}")
print()
for old, new in STATUS_MIGRATIONS.items():
    n = counts_by_old.get(old, 0)
    print(f"  {old:14s} → {new:10s}  {n} task(s)")
    for title, oid in samples_by_old[old][:5]:
        print(f"      OID {oid:5d}  {title[:80]}")
    if n > 5:
        print(f"      … and {n - 5} more")
    print()

## Step 3 — Apply the migrationRuns the actual write. Updates are batched in groups of 500 to stay underAGOL's per-request limits. Each successful batch prints its count; failuresare listed at the end so you can investigate without losing context.**This is the write step.** If Step 2 looked wrong, stop here and reach outto Peter before running this cell.

In [ ]:
if not to_update:
    print("Nothing to migrate — all task statuses are already canonical.")
else:
    BATCH = 500
    total_ok = 0
    total_fail = 0
    failures = []

    for i in range(0, len(to_update), BATCH):
        batch = to_update[i:i+BATCH]
        result = tasks.edit_features(updates=batch)
        for r in (result.get('updateResults') or []):
            if r.get('success'):
                total_ok += 1
            else:
                total_fail += 1
                failures.append(r)
        print(f"  batch {i//BATCH + 1}: {len(batch)} attempted, {total_ok} cumulative success")

    print()
    print(f"Migration complete. Successful updates: {total_ok}. Failed: {total_fail}.")
    if failures:
        print()
        print("Failed updates (first 5):")
        for f in failures[:5]:
            print(f"  {f}")

## Step 4 — VerifyRe-queries the task layer and prints the new status distribution. If themigration succeeded, none of the deprecated values (`In Progress`, `NotStarted`, `Pending`, `Completed`) should appear in the list — only thecanonical seven (Planned, Scheduled, Active, Waiting for Response, On Hold,Complete, Canceled) plus any non-canonical values that pre-existed andweren't part of this migration.

In [ ]:
after = tasks.query(where="deleted_at IS NULL", out_fields="status", return_geometry=False).features
after_counts = Counter(f.attributes.get('status') or '(blank)' for f in after)

print("Post-migration status distribution:")
for status, n in sorted(after_counts.items(), key=lambda kv: (-kv[1], kv[0])):
    flag = '  ⚠ deprecated — re-run?' if status in STATUS_MIGRATIONS else ''
    print(f"  {n:5d}  {status}{flag}")

leftover = sum(after_counts.get(k, 0) for k in STATUS_MIGRATIONS.keys())
print()
if leftover == 0:
    print("✓ All deprecated statuses are gone.")
else:
    print(f"⚠ {leftover} task(s) still carry a deprecated status. Re-run Step 3.")

## DoneIf verification shows zero deprecated statuses, the migration is complete.The app will now show the canonical seven everywhere — pill colors, filterchips, sort orders, dropdowns.The `agolTaskToLocal` translation layer in [src/agol.js](../src/agol.js) isstill safe to keep in place as a belt-and-suspenders measure; it just won'thave anything to translate going forward.